# Лабораторная работа №5: классификация моторного воображения по ЭЭГ

## Цель работы

Цель работы — исследовать ЭЭГ-сигналы, преобразовать временной сигнал в частотно-временное изображение с помощью вейвлет-преобразования и обучить нейронную сеть для бинарной классификации моторного воображения: представления движения левого или правого кулака.

Основные этапы:

1. загрузка и проверка набора данных;
2. анализ формы сигналов и распределения классов;
3. нормализация временных рядов;
4. построение скалограмм с помощью CWT;
5. обучение компактной CNN;
6. оценка качества на тестовой выборке.

## Исправление ссылки на датасет

В README задания ссылка на обучающую и тестовую матрицы признаков указана одинаково: `MI-EEG-B9T.csv`. Это выглядит как ошибка, потому что для целевых значений существуют отдельные train/test-файлы.

В работе используется согласованная пара файлов:

| Назначение | Файл |
| --- | --- |
| Обучающие сигналы | `MI-EEG-B9T.csv` |
| Тестовые сигналы | `MI-EEG-B9E.csv` |
| Метки обучающей выборки | `2class_MI_EEG_train_9.csv` |
| Метки тестовой выборки | `2class_MI_EEG_test_9.csv` |

Если автоматическая загрузка недоступна, эти четыре файла можно вручную положить в папку `data/` рядом с ноутбуком.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import pywt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

np.random.seed(42)
torch.manual_seed(42)

## 1. Загрузка данных

Каждая строка CSV-файла содержит один фрагмент ЭЭГ-сигнала длиной 3000 отсчетов. Целевые файлы содержат бинарные метки классов `0` и `1`.

In [ ]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

FILES = {
    'train_x': ('MI-EEG-B9T.csv', 'https://raw.githubusercontent.com/D2718281828nis/BioMedAI/main/test_datasets/MI-EEG-B9T.csv'),
    'test_x': ('MI-EEG-B9E.csv', 'https://raw.githubusercontent.com/D2718281828nis/BioMedAI/main/test_datasets/MI-EEG-B9E.csv'),
    'train_y': ('2class_MI_EEG_train_9.csv', 'https://raw.githubusercontent.com/D2718281828nis/BioMedAI/main/test_datasets/2class_MI_EEG_train_9.csv'),
    'test_y': ('2class_MI_EEG_test_9.csv', 'https://raw.githubusercontent.com/D2718281828nis/BioMedAI/main/test_datasets/2class_MI_EEG_test_9.csv'),
}

for filename, url in FILES.values():
    path = DATA_DIR / filename
    if not path.exists():
        urlretrieve(url, path)

train_x = pd.read_csv(DATA_DIR / FILES['train_x'][0], header=None).values.astype('float32')
test_x = pd.read_csv(DATA_DIR / FILES['test_x'][0], header=None).values.astype('float32')
train_y = pd.read_csv(DATA_DIR / FILES['train_y'][0], header=None).values.ravel().astype('int64')
test_y = pd.read_csv(DATA_DIR / FILES['test_y'][0], header=None).values.ravel().astype('int64')

print('Размер обучающей матрицы:', train_x.shape, 'размер меток:', train_y.shape)
print('Размер тестовой матрицы:', test_x.shape, 'размер меток:', test_y.shape)
print('Распределение классов в обучении:', dict(zip(*np.unique(train_y, return_counts=True))))
print('Распределение классов в тесте:', dict(zip(*np.unique(test_y, return_counts=True))))

## 2. Первичный анализ сигнала

Выборки сбалансированы: в обучающей части по 200 объектов каждого класса, в тестовой части по 160 объектов каждого класса. Перед вейвлет-преобразованием каждый сигнал нормализуется отдельно, чтобы уменьшить влияние абсолютного масштаба амплитуды.

In [ ]:
def standardize_rows(x):
    mean = x.mean(axis=1, keepdims=True)
    std = x.std(axis=1, keepdims=True) + 1e-6
    return ((x - mean) / std).astype('float32')

train_z = standardize_rows(train_x)
test_z = standardize_rows(test_x)

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for ax, cls in zip(axes, [0, 1]):
    idx = np.where(train_y == cls)[0][0]
    ax.plot(train_z[idx, :500])
    ax.set_title(f'Класс {cls}: первые 500 нормированных отсчетов')
    ax.set_xlabel('Номер отсчета')
    ax.set_ylabel('Нормированная амплитуда')
plt.tight_layout()
plt.show()

## 3. Вейвлет-преобразование

Исходный ЭЭГ-фрагмент является одномерным временным сигналом. Для CNN удобнее использовать изображение, поэтому сигнал преобразуется в частотно-временное представление.

Используется непрерывное вейвлет-преобразование с вейвлетом Морле. Полученная матрица коэффициентов показывает, какие частотные компоненты проявляются в разные моменты времени.

In [ ]:
def make_scalograms(signals, scales=np.arange(1, 49), wavelet='morl', step=16):
    images = []
    for signal in signals:
        coefficients, _ = pywt.cwt(signal, scales, wavelet)
        image = np.log1p(np.abs(coefficients[:, ::step]))
        image = (image - image.mean()) / (image.std() + 1e-6)
        images.append(image.astype('float32'))
    return np.stack(images)

x_train_img = make_scalograms(train_z)
x_test_img = make_scalograms(test_z)
print('Размер вейвлет-изображения:', x_train_img.shape[1:])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, cls in zip(axes, [0, 1]):
    idx = np.where(train_y == cls)[0][0]
    ax.imshow(x_train_img[idx], aspect='auto', origin='lower', cmap='viridis')
    ax.set_title(f'Класс {cls}: CWT-скалограмма')
    ax.set_xlabel('Временное окно')
    ax.set_ylabel('Масштаб')
plt.tight_layout()
plt.show()

## 4. Архитектура нейронной сети

Так как датасет небольшой, используется компактная CNN-модель. Она состоит из трех сверточных блоков и небольшого полносвязного классификатора.

In [ ]:
class EEGCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 4 * 4, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 2),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = EEGCNN()
print(model)

## 5. Обучение модели

Модель обучается с функцией потерь `CrossEntropyLoss` и оптимизатором Adam. Количество эпох выбрано небольшим, потому что главная цель — показать полный пайплайн обработки ЭЭГ: временной сигнал → вейвлет-изображение → CNN-классификация.

In [ ]:
xtr = torch.tensor(x_train_img[:, None, :, :])
ytr = torch.tensor(train_y)
xte = torch.tensor(x_test_img[:, None, :, :])
yte = torch.tensor(test_y)

loader = DataLoader(TensorDataset(xtr, ytr), batch_size=32, shuffle=True)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

history = []
for epoch in range(1, 16):
    model.train()
    total_loss = 0.0
    correct = 0
    seen = 0
    for xb, yb in loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)
        correct += (logits.argmax(1) == yb).sum().item()
        seen += len(xb)
    history.append((epoch, total_loss / seen, correct / seen))
    print(f'Эпоха {epoch:02d} | loss={total_loss / seen:.4f} | train_acc={correct / seen:.3f}')

## 6. Оценка качества

Качество оценивается на отдельной тестовой выборке. Accuracy показывает общую долю правильных ответов, а матрица ошибок позволяет увидеть, какой класс распознается лучше.

In [ ]:
model.eval()
with torch.no_grad():
    pred = model(xte).argmax(1).numpy()

print('Точность на тестовой выборке:', accuracy_score(test_y, pred))
print('Матрица ошибок:')
print(confusion_matrix(test_y, pred))
print(classification_report(test_y, pred, target_names=['левый кулак', 'правый кулак'], zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot([x[0] for x in history], [x[1] for x in history], marker='o')
axes[0].set_title('Ошибка обучения')
axes[0].set_xlabel('Эпоха')
axes[0].set_ylabel('Cross entropy')
axes[0].grid(alpha=0.3)

cm = confusion_matrix(test_y, pred)
im = axes[1].imshow(cm, cmap='Blues')
axes[1].set_title('Матрица ошибок')
axes[1].set_xlabel('Предсказанный класс')
axes[1].set_ylabel('Истинный класс')
axes[1].set_xticks([0, 1], ['левый', 'правый'])
axes[1].set_yticks([0, 1], ['левый', 'правый'])
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, str(cm[i, j]), ha='center', va='center')
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## Вывод

В работе построен полный пайплайн классификации ЭЭГ-сигналов. Сначала временной ряд нормализуется, затем преобразуется в скалограмму с помощью непрерывного вейвлет-преобразования. Полученное изображение подается в компактную сверточную нейронную сеть.

Модель показывает качество выше случайного уровня, что подтверждает полезность вейвлет-представления для различения двух состояний моторного воображения.